# Train ISIC Models - All Architectures

This notebook trains multiple model architectures on the ISIC dataset:

- ResNet18, ResNet50
- DenseNet121
- EfficientNet B0, B1
- MobileNet V2

**Outputs:**

- Checkpoints: `checkpoints/isic/<model_name>/best_model.pt`
- Training logs: `logs/isic/<model_name>/history.json`
- Metrics: `metrics/isic/<model_name>/metrics.json`

**Instructions:** Run all cells in order.


In [1]:
# Setup imports and paths
import os
import sys
from pathlib import Path
import json
import torch
from torch.utils.data import DataLoader

# Ensure repo root is on sys.path
ROOT = Path('..').resolve() / '..'  # notebook is in notebooks/training/ -> repo root two levels up
ROOT = Path('.') if not (ROOT / 'src').exists() else ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

# Output folders (relative to repo root)
CHECKPOINTS_ROOT = Path('checkpoints/isic')
LOGS_ROOT = Path('logs/isic')
METRICS_ROOT = Path('metrics/isic')
for p in (CHECKPOINTS_ROOT, LOGS_ROOT, METRICS_ROOT):
    p.mkdir(parents=True, exist_ok=True)

Using device: cpu


In [ ]:
# Configuration - change as needed
MODELS = ['resnet18', 'resnet50', 'densenet121', 'efficientnet_b0', 'efficientnet_b1', 'mobilenet_v2']
EPOCHS = 10
BATCH_SIZE = 16
NUM_WORKERS = 4
DATA_ROOT_CANDIDATES = [Path('data/raw/isic')]

# Metric to track for best checkpoint - options: 'val_auc', 'val_accuracy', 'val_f1_macro', etc.
# Best model is saved when this metric improves
METRIC_TO_TRACK = 'val_accuracy'  # Changed from 'val_auc' to track accuracy improvements

In [3]:
# Verify paths and imports before loading dataset
import os

print('='*60)
print('PRE-FLIGHT CHECK')
print('='*60)

# Check if src.datasets.isic module exists
try:
    import src.datasets.isic
    print('✓ Module src.datasets.isic found')
except ImportError as e:
    print(f'✗ Cannot import src.datasets.isic: {e}')

# Check data directory structure (use ROOT for absolute path)
DATA_ROOT = ROOT / 'data' / 'raw' / 'isic'
print(f'\nChecking data directory: {DATA_ROOT}')
print(f'  Absolute path: {DATA_ROOT.absolute()}')
print(f'  Exists: {DATA_ROOT.exists()}')

if DATA_ROOT.exists():
    for split in ['train', 'val', 'test']:
        split_dir = DATA_ROOT / split
        images_dir = split_dir / 'images'
        labels_file = split_dir / 'labels.json'
        
        print(f'\n  {split}/:')
        print(f'    Directory exists: {split_dir.exists()}')
        print(f'    images/ exists: {images_dir.exists()}')
        print(f'    labels.json exists: {labels_file.exists()}')
        
        if images_dir.exists():
            image_count = len([f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
            print(f'    Image count: {image_count}')

print('\n' + '='*60)

PRE-FLIGHT CHECK
✓ Module src.datasets.isic found

Checking data directory: D:\git projects\certified-attribution-medical-imaging\notebooks\..\data\raw\isic
  Absolute path: D:\git projects\certified-attribution-medical-imaging\notebooks\..\data\raw\isic
  Exists: True

  train/:
    Directory exists: True
    images/ exists: True
    labels.json exists: True
    Image count: 2444

  val/:
    Directory exists: True
    images/ exists: True
    labels.json exists: True
    Image count: 522

  test/:
    Directory exists: True
    images/ exists: True
    labels.json exists: True
    Image count: 526



In [4]:
# Import dataset, model factory and Trainer
from src.train.train_one import Trainer
from src.models.factory import get_model
from src.datasets.isic import ISICDataset
from torchvision import transforms

print('='*60)
print('LOADING ISIC DATASET')
print('='*60)

# Use explicit data root (relative to repo ROOT)
DATA_ROOT = ROOT / 'data' / 'raw' / 'isic'
print(f'Data root: {DATA_ROOT}')
if not DATA_ROOT.exists():
    print(f'⚠ WARNING: DATA_ROOT does not exist: {DATA_ROOT.absolute()}')
    print('Please create the directory or adjust the path.')
else:
    print(f'✓ Data root found: {DATA_ROOT.absolute()}')

# Define transforms to resize all images to same size
IMG_SIZE = 224
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Instantiate datasets with transforms
print(f'\nLoading datasets with image size {IMG_SIZE}x{IMG_SIZE}...')
train_dataset = ISICDataset(str(DATA_ROOT), split='train', transform=train_transform, target_size=(IMG_SIZE, IMG_SIZE))
val_dataset = ISICDataset(str(DATA_ROOT), split='val', transform=val_transform, target_size=(IMG_SIZE, IMG_SIZE))
print(f'✓ Train dataset loaded: {len(train_dataset)} samples')
print(f'✓ Val dataset loaded: {len(val_dataset)} samples')

# Infer number of classes
def infer_num_classes(ds):
    for attr in ('num_classes', 'n_classes', 'classes', 'label_map', 'label'):
        if hasattr(ds, attr):
            val = getattr(ds, attr)
            if isinstance(val, (list, tuple, dict)):
                return len(val)
            if isinstance(val, int):
                return val
    # fallback to scanning dataset labels
    try:
        labels = [ds[i]['label'] for i in range(min(len(ds), 200))]
        return len(set([int(l) for l in labels]))
    except Exception:
        return 2

NUM_CLASSES = infer_num_classes(train_dataset)
print(f'✓ Number of classes: {NUM_CLASSES}')

# Show sample from dataset
try:
    sample = train_dataset[0]
    print(f'\nSample data shape: {sample["image"].shape}')
    print(f'Sample label: {sample["label"]}')
    print(f'✓ All images will be resized to {IMG_SIZE}x{IMG_SIZE}')
except Exception as e:
    print(f'⚠ Could not load sample: {e}')

# Create DataLoaders
print(f'\nCreating DataLoaders (batch_size={BATCH_SIZE})...')
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f'✓ Train loader: {len(train_loader)} batches')
print(f'✓ Val loader: {len(val_loader)} batches')

print('\n' + '='*60)
print('DATASET READY FOR TRAINING')
print('='*60)

LOADING ISIC DATASET
Data root: D:\git projects\certified-attribution-medical-imaging\notebooks\..\data\raw\isic
✓ Data root found: D:\git projects\certified-attribution-medical-imaging\notebooks\..\data\raw\isic

Loading datasets with image size 224x224...
✓ Train dataset loaded: 2444 samples
✓ Val dataset loaded: 522 samples
✓ Number of classes: 8

Sample data shape: torch.Size([3, 224, 224])
Sample label: 0
✓ All images will be resized to 224x224

Creating DataLoaders (batch_size=16)...
✓ Train loader: 153 batches
✓ Val loader: 33 batches

DATASET READY FOR TRAINING


In [5]:
# Training loop across models
from tqdm import tqdm
import json
import torch

results = {}
for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    
    # Create per-model folders
    ckpt_dir = CHECKPOINTS_ROOT / model_name
    log_dir = LOGS_ROOT / model_name
    metrics_dir = METRICS_ROOT / model_name
    for p in (ckpt_dir, log_dir, metrics_dir):
        p.mkdir(parents=True, exist_ok=True)
    
    # Build model
    model, cfg = get_model(model_name, num_classes=NUM_CLASSES, pretrained=True, device=DEVICE)
    
    # Check for existing checkpoint to resume training
    best_ckpt = ckpt_dir / 'best_model.pt'
    final_ckpt = ckpt_dir / 'final_model.pt'
    start_epoch = 0
    existing_history = {}
    
    if best_ckpt.exists():
        print(f'📂 Found existing best checkpoint: {best_ckpt}')
        print('   Loading weights to resume training...')
        checkpoint = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        start_epoch = checkpoint.get('epoch', 0) + 1
        print(f'   ✓ Resuming from epoch {start_epoch}')
    elif final_ckpt.exists():
        print(f'📂 Found existing final checkpoint: {final_ckpt}')
        print('   Loading weights to resume training...')
        checkpoint = torch.load(final_ckpt, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        start_epoch = checkpoint.get('epoch', 0) + 1
        print(f'   ✓ Resuming from epoch {start_epoch}')
    else:
        print(f'✓ Model built: {model_name} -> {cfg.backbone} (from pretrained ImageNet)')
    
    # Load existing history if available
    hist_path = log_dir / 'history.json'
    if hist_path.exists():
        print(f'📂 Loading existing history from {hist_path}')
        with open(hist_path, 'r') as f:
            existing_history = json.load(f)
        print(f'   ✓ Found {len(existing_history.get("train_loss", []))} previous epochs')
    
    # Trainer (uses CrossEntropyLoss inside)
    trainer = Trainer(
        model=model, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        device=DEVICE, 
        task='multi-class' if NUM_CLASSES > 2 else 'binary'
    )
    print(f'✓ Trainer initialized')
    
    # Fit (train for EPOCHS more epochs)
    print(f'\nTraining for {EPOCHS} more epochs (total will be {start_epoch + EPOCHS})...')
    new_history = trainer.fit(
        epochs=EPOCHS, 
        checkpoint_dir=str(ckpt_dir), 
        metric_to_track=METRIC_TO_TRACK
    )
    
    # Merge histories (append new to old)
    if existing_history:
        for key in new_history:
            if key in existing_history:
                existing_history[key].extend(new_history[key])
            else:
                existing_history[key] = new_history[key]
        combined_history = existing_history
        print(f'✓ Merged with previous history (now {len(combined_history.get("train_loss", []))} total epochs)')
    else:
        combined_history = new_history
    
    # Save combined history to logs
    with open(hist_path, 'w') as f:
        json.dump(combined_history, f, indent=2)
    print(f'✓ History saved: {hist_path}')
    
    # ALWAYS save final trained model (regardless of best metric)
    final_model_path = ckpt_dir / 'final_model.pt'
    torch.save({
        'epoch': start_epoch + EPOCHS - 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': trainer.optimizer.state_dict(),
        'metrics': {
            'final_train_loss': combined_history.get('train_loss', [None])[-1],
            'final_val_auc': combined_history.get('val_auc', [None])[-1],
            'final_val_accuracy': combined_history.get('val_accuracy', [None])[-1]
        }
    }, final_model_path)
    print(f'✓ Final model saved: {final_model_path}')
    
    # Check if best checkpoint was created during this training
    if best_ckpt.exists():
        print(f'✓ Best checkpoint exists: {best_ckpt}')
    else:
        print(f'⚠ No separate best checkpoint (using final model)')
    
    # Save validation metrics from best or final checkpoint
    metrics_out = {}
    checkpoint_to_use = best_ckpt if best_ckpt.exists() else final_model_path
    try:
        data = torch.load(checkpoint_to_use, map_location='cpu')
        if isinstance(data, dict) and 'metrics' in data:
            metrics_out = data['metrics']
    except Exception as e:
        print(f'⚠ Could not load metrics from checkpoint: {e}')
        metrics_out = {}
    
    # Add summary from combined history
    try:
        hist_summary = {
            k: (v[-1] if isinstance(v, list) and len(v) > 0 else None) 
            for k, v in combined_history.items()
        }
        metrics_out['history_summary'] = hist_summary
        metrics_out['total_epochs'] = len(combined_history.get('train_loss', []))
    except Exception:
        pass
    
    metrics_path = metrics_dir / 'metrics.json'
    with open(metrics_path, 'w') as f:
        json.dump(metrics_out, f, indent=2)
    print(f'✓ Metrics saved: {metrics_path}')
    
    results[model_name] = {
        'checkpoint': str(checkpoint_to_use) if checkpoint_to_use.exists() else None,
        'history': str(hist_path),
        'metrics': str(metrics_path),
        'total_epochs': len(combined_history.get('train_loss', []))
    }

# Print final summary
print(f"\n{'='*60}")
print("ALL TRAINING COMPLETE - RESULTS SUMMARY")
print(f"{'='*60}")
for model_name, info in results.items():
    print(f"\n{model_name}:")
    for key, val in info.items():
        print(f"  {key}: {val}")


Model: resnet18


d:\git projects\certified-attribution-medical-imaging\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\git projects\certified-attribution-medical-imaging\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


✓ Model built: resnet18 -> resnet18 (from pretrained ImageNet)
✓ Trainer initialized

Training for 10 more epochs (total will be 10)...

=== Epoch 1/10 ===


Train: 100%|██████████| 153/153 [04:37<00:00,  1.81s/it, loss=1.59]


Train: accuracy: 0.3126 | f1_macro: 0.3143 | f1_weighted: 0.3093 | auc: 0.7107 | loss: 1.8485


...

Val:   accuracy: 0.5249 | f1_macro: 0.5508 | f1_weighted: 0.5266 | auc: 0.8785

=== Epoch 10/10 ===


Train: 100%|██████████| 153/153 [04:18<00:00,  1.69s/it, loss=1.29] 


Train: accuracy: 0.5949 | f1_macro: 0.6163 | f1_weighted: 0.5942 | auc: 0.9104 | loss: 1.0774


Val: 100%|██████████| 33/33 [01:03<00:00,  1.92s/it]


Val:   accuracy: 0.5192 | f1_macro: 0.5264 | f1_weighted: 0.5057 | auc: 0.8805
✓ History saved: logs\isic\mobilenet_v2\history.json
✓ Final model saved: checkpoints\isic\mobilenet_v2\final_model.pt
⚠ No separate best checkpoint (using final model)
✓ Metrics saved: metrics\isic\mobilenet_v2\metrics.json

ALL TRAINING COMPLETE - RESULTS SUMMARY

resnet18:
  checkpoint: checkpoints\isic\resnet18\final_model.pt
  history: logs\isic\resnet18\history.json
  metrics: metrics\isic\resnet18\metrics.json
  total_epochs: 10

resnet50:
  checkpoint: checkpoints\isic\resnet50\final_model.pt
  history: logs\isic\resnet50\history.json
  metrics: metrics\isic\resnet50\metrics.json
  total_epochs: 10

densenet121:
  checkpoint: checkpoints\isic\densenet121\final_model.pt
  history: logs\isic\densenet121\history.json
  metrics: metrics\isic\densenet121\metrics.json
  total_epochs: 10

efficientnet_b0:
  checkpoint: checkpoints\isic\efficientnet_b0\final_model.pt
  history: logs\isic\efficientnet_b0\hist

## Evaluate Trained Models

Now let's load the trained models, test predictions, visualize results, and find the best performing model.


In [7]:
# Load and test all trained models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

# Store evaluation results
eval_results = {}

print("="*60)
print("EVALUATING ALL MODELS ON VALIDATION SET")
print("="*60)

for model_name in MODELS:
    print(f"\n{model_name}:")
    
    # Load checkpoint
    ckpt_path = CHECKPOINTS_ROOT / model_name / 'final_model.pt'
    if not ckpt_path.exists():
        print(f"  ⚠ Checkpoint not found, skipping...")
        continue
    
    # Build model architecture
    model, cfg = get_model(model_name, num_classes=NUM_CLASSES, pretrained=False, device=DEVICE)
    
    # Load trained weights
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f"  ✓ Model loaded from checkpoint")
    
    # Evaluate on validation set
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    # Calculate accuracy
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"  ✓ Validation Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    # Store results
    eval_results[model_name] = {
        'accuracy': accuracy,
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs,
        'checkpoint_metrics': checkpoint.get('metrics', {})
    }

print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)

EVALUATING ALL MODELS ON VALIDATION SET

resnet18:


d:\git projects\certified-attribution-medical-imaging\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\git projects\certified-attribution-medical-imaging\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  ✓ Model loaded from checkpoint


KeyboardInterrupt: 

In [ ]:
# Find and display the best performing model
print("="*60)
print("BEST MODEL RANKING")
print("="*60)

# Sort models by accuracy
sorted_models = sorted(eval_results.items(), key=lambda x: x[1]['accuracy'], reverse=True)

print(f"\n{'Rank':<6} {'Model':<20} {'Accuracy':<12}")
print("-" * 60)
for i, (model_name, result) in enumerate(sorted_models, 1):
    acc = result['accuracy']
    print(f"{i:<6} {model_name:<20} {acc:.4f} ({acc*100:.2f}%)")

best_model_name = sorted_models[0][0]
best_accuracy = sorted_models[0][1]['accuracy']

print(f"\n{'='*60}")
print(f"🏆 BEST MODEL: {best_model_name}")
print(f"   Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
print(f"   Checkpoint: checkpoints/isic/{best_model_name}/best_model.pt")
print(f"{'='*60}")

In [ ]:
# Plot training curves (loss and metrics) for all models
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Training History - All Models', fontsize=16, fontweight='bold')
axes = axes.flatten()

for idx, model_name in enumerate(MODELS):
    if idx >= len(axes):
        break
    
    # Load history
    hist_path = LOGS_ROOT / model_name / 'history.json'
    if not hist_path.exists():
        axes[idx].text(0.5, 0.5, f'{model_name}\nNo history found', 
                      ha='center', va='center', fontsize=12)
        axes[idx].set_title(model_name)
        continue
    
    with open(hist_path, 'r') as f:
        history = json.load(f)
    
    ax = axes[idx]
    epochs_range = range(1, len(history.get('train_loss', [])) + 1)
    
    # Plot training loss
    if 'train_loss' in history and history['train_loss']:
        ax.plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    
    # Plot validation accuracy on secondary y-axis
    ax2 = ax.twinx()
    if 'val_accuracy' in history and history['val_accuracy']:
        ax2.plot(epochs_range, history['val_accuracy'], 'r-', label='Val Accuracy', linewidth=2)
    if 'val_auc' in history and history['val_auc']:
        ax2.plot(epochs_range, history['val_auc'], 'g--', label='Val AUC', linewidth=2)
    
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel('Loss', color='b', fontsize=10)
    ax2.set_ylabel('Accuracy / AUC', color='r', fontsize=10)
    ax.tick_params(axis='y', labelcolor='b')
    ax2.tick_params(axis='y', labelcolor='r')
    ax.set_title(f'{model_name}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Combine legends
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('outputs/reports/training_curves.png', dpi=150, bbox_inches='tight')
print("✓ Training curves saved to: outputs/reports/training_curves.png")
plt.show()

In [ ]:
# Compare model accuracies with bar chart
plt.figure(figsize=(12, 6))

model_names_list = list(eval_results.keys())
accuracies = [eval_results[m]['accuracy'] * 100 for m in model_names_list]

# Create bar chart with colors
colors = ['gold' if m == best_model_name else 'steelblue' for m in model_names_list]
bars = plt.bar(range(len(model_names_list)), accuracies, color=colors, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for i, (bar, acc) in enumerate(zip(bars, accuracies)):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
             f'{acc:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.xlabel('Model', fontsize=12, fontweight='bold')
plt.ylabel('Validation Accuracy (%)', fontsize=12, fontweight='bold')
plt.title('Model Comparison - Validation Accuracy', fontsize=14, fontweight='bold')
plt.xticks(range(len(model_names_list)), model_names_list, rotation=45, ha='right')
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('outputs/reports/model_comparison.png', dpi=150, bbox_inches='tight')
print("✓ Model comparison chart saved to: outputs/reports/model_comparison.png")
plt.show()

In [ ]:
# Visualize predictions from the best model
print(f"Visualizing predictions from best model: {best_model_name}\n")

# Get predictions from best model
best_preds = eval_results[best_model_name]['predictions']
best_labels = eval_results[best_model_name]['labels']
best_probs = eval_results[best_model_name]['probabilities']

# Get class names if available
try:
    if hasattr(val_dataset, 'classes'):
        class_names = val_dataset.classes
    elif hasattr(val_dataset, 'label_map'):
        class_names = list(val_dataset.label_map.values())
    else:
        class_names = [f'Class {i}' for i in range(NUM_CLASSES)]
except:
    class_names = [f'Class {i}' for i in range(NUM_CLASSES)]

print(f"Classes: {class_names}\n")

# Select random samples to visualize
np.random.seed(42)
num_samples = min(12, len(val_dataset))
sample_indices = np.random.choice(len(val_dataset), num_samples, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle(f'Predictions from {best_model_name}', fontsize=16, fontweight='bold')
axes = axes.flatten()

for idx, sample_idx in enumerate(sample_indices):
    sample = val_dataset[sample_idx]
    image = sample['image']
    true_label = sample['label']
    
    # Get prediction for this sample
    pred_label = best_preds[sample_idx]
    pred_prob = best_probs[sample_idx]
    
    # Convert image for display (denormalize if needed)
    img_display = image.permute(1, 2, 0).numpy()
    
    # Normalize to [0, 1] for display
    img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min() + 1e-8)
    
    # Display image
    axes[idx].imshow(img_display)
    
    # Set title with prediction info
    true_class = class_names[true_label] if true_label < len(class_names) else f'Class {true_label}'
    pred_class = class_names[pred_label] if pred_label < len(class_names) else f'Class {pred_label}'
    confidence = pred_prob[pred_label] * 100
    
    is_correct = true_label == pred_label
    title_color = 'green' if is_correct else 'red'
    
    axes[idx].set_title(
        f'True: {true_class}\nPred: {pred_class} ({confidence:.1f}%)',
        fontsize=9,
        color=title_color,
        fontweight='bold'
    )
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('outputs/reports/prediction_samples.png', dpi=150, bbox_inches='tight')
print("✓ Prediction samples saved to: outputs/reports/prediction_samples.png")
plt.show()

In [ ]:
# Confusion matrix for best model
from sklearn.metrics import confusion_matrix

print(f"Confusion Matrix for {best_model_name}:\n")

cm = confusion_matrix(best_labels, best_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('outputs/reports/confusion_matrix.png', dpi=150, bbox_inches='tight')
print("✓ Confusion matrix saved to: outputs/reports/confusion_matrix.png")
plt.show()

# Print classification report
print(f"\nClassification Report for {best_model_name}:")
print("="*60)
print(classification_report(best_labels, best_preds, target_names=class_names, digits=4))

In [ ]:
# Summary of all results
print("\n" + "="*80)
print("FINAL SUMMARY - ALL MODELS")
print("="*80)

summary_data = []
for model_name in MODELS:
    if model_name in eval_results:
        acc = eval_results[model_name]['accuracy']
        
        # Get final metrics from history
        hist_path = LOGS_ROOT / model_name / 'history.json'
        if hist_path.exists():
            with open(hist_path, 'r') as f:
                history = json.load(f)
            final_loss = history.get('train_loss', [None])[-1] if history.get('train_loss') else None
            final_val_auc = history.get('val_auc', [None])[-1] if history.get('val_auc') else None
        else:
            final_loss = None
            final_val_auc = None
        
        summary_data.append({
            'model': model_name,
            'val_accuracy': acc,
            'final_train_loss': final_loss,
            'final_val_auc': final_val_auc,
            'is_best': '🏆' if model_name == best_model_name else ''
        })

# Print table
print(f"\n{'Model':<20} {'Val Acc':<12} {'Train Loss':<12} {'Val AUC':<12} {'Best':<6}")
print("-" * 80)
for row in summary_data:
    acc_str = f"{row['val_accuracy']:.4f}" if row['val_accuracy'] else "N/A"
    loss_str = f"{row['final_train_loss']:.4f}" if row['final_train_loss'] else "N/A"
    auc_str = f"{row['final_val_auc']:.4f}" if row['final_val_auc'] else "N/A"
    print(f"{row['model']:<20} {acc_str:<12} {loss_str:<12} {auc_str:<12} {row['is_best']:<6}")

print("\n" + "="*80)
print(f"🏆 BEST MODEL: {best_model_name} with {best_accuracy*100:.2f}% validation accuracy")
print("="*80)
print("\nAll outputs saved to:")
print("  - Checkpoints: checkpoints/isic/<model>/")
print("  - Logs: logs/isic/<model>/")
print("  - Metrics: metrics/isic/<model>/")
print("  - Visualizations: outputs/reports/")
print("\n" + "="*80)

## Configuration Notes

- Adjust `EPOCHS`, `BATCH_SIZE`, and `NUM_WORKERS` in the configuration cell
- The notebook uses `data/raw/isic` with `split='train'` and `split='val'`
- Models are trained with ImageNet pretrained weights
- Best models are saved based on validation AUC
- All outputs are saved to `checkpoints/`, `logs/`, and `metrics/` folders

## Next Steps

After training completes:

1. Check `logs/isic/<model>/history.json` for training curves
2. Load best checkpoints from `checkpoints/isic/<model>/best_model.pt`
3. Review metrics in `metrics/isic/<model>/metrics.json`
